#Silver Layer Product

In [0]:
import pyspark.sql.functions as F
#all necessary libraries are imported

#simple dataframe read to see the bronze delta table
df_product = spark.table("accenture_final_project.bronze_layer.product")
display(df_product)


###Checking the distinct values of the string columns

In [0]:
display(df_product.select("Product").distinct().orderBy("product"))

In [0]:
display(df_product.select("Color").distinct())

In [0]:
display(df_product.select("Subcategory").distinct())

In [0]:
display(df_product.select("Category").distinct())

In [0]:
display(df_product.select("Background_Color_Format").distinct())

In [0]:
display(df_product.select("Font_Color_Format").distinct())

In [0]:
# Calculating nulls for each column
null_counts = df_product.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_product.columns
])

print("Null count per column:")
display(null_counts)

###Cleaning the data

In [0]:


df_product_silver = (
    df_product.dropDuplicates()
    # 1 Removing whitespace from all string type columns 
    .withColumn("Product", F.trim(F.col("Product")))
    .withColumn("Color", F.trim(F.col("Color")))
    .withColumn("subcategory", F.trim(F.col("subcategory")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("background_color_format", F.trim(F.col("background_color_format")))
    .withColumn("font_color_format", F.trim(F.col("font_color_format")))
    .withColumn("standard_cost", F.trim(F.col("standard_cost")))
    # 2 Removing the $ sign and , from the StandardCost column and casting to double
    .withColumn("standard_cost", F.regexp_replace("standard_cost", r"[\$,]", "").cast("double"))
    # 3 Removing all negative/zero standard cost values
    .filter(F.col("standard_cost") > 0) 
    # 4 Renaming incorrect values based on the above investigation for distinct values
    .withColumn("Color", 
        F.when(F.col("Color") == "MULTI", "Multi")
         .when(F.col("Color") == "RED", "Red")
         .when(F.col("Color") == "N/A", "Unknown Color")
         .when(F.col("Color") == "YELLOW", "Yellow")
         .otherwise(F.col("Color")) # keeps all other colors as they are
    )
    .withColumn("Subcategory", F.when(F.col("Subcategory") == "Whels", "Wheels").otherwise(F.col("Subcategory")) )
    # 5 Renaming columns
    .withColumnRenamed("Standard_Cost", "StandardCost")
    .withColumnRenamed("Background_Color_Format", "BackgroundColorFormat")
    .withColumnRenamed("Font_Color_Format", "FontColorFormat")
).orderBy("ProductKey")


display(df_product_silver)

##Save the data into a Delta table

In [0]:
# We save the DataFrame as a delta table 
(
df_product_silver.write.mode("overwrite")
                       .option("overwriteSchema", "true")
                       .format("delta")
                       .saveAsTable("accenture_final_project.silver_layer.product")
)

